In [1]:
import json
from pathlib import Path
from datetime import datetime
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "variant_03"
NORMALIZED_DIR = PROJECT_ROOT / "data" / "normalized" / "variant_03"

print("Текущая рабочая папка:", Path.cwd())
print("Корень проекта:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("NORMALIZED_DIR:", NORMALIZED_DIR)


def get_latest_json_file(folder: Path) -> Path:
    files = list(folder.glob("*.json"))

    if not files:
        raise FileNotFoundError(f"В папке нет json-файлов: {folder}")

    latest_file = max(files, key=lambda f: f.stat().st_mtime)
    return latest_file


def main():
    if not RAW_DIR.exists():
        raise FileNotFoundError(f"Папка raw не найдена: {RAW_DIR}")

    raw_path = get_latest_json_file(RAW_DIR)
    print("Использую raw file:", raw_path)

    with open(raw_path, encoding="utf-8") as f:
        data = json.load(f)

    df = pd.DataFrame({
        "time": data["hourly"]["time"],
        "temperature": data["hourly"]["temperature_2m"]
    })

    print("Исходный размер:", df.shape)
    print("Исходные типы:")
    print(df.dtypes)

    df["time"] = pd.to_datetime(df["time"])
    df["temperature"] = pd.to_numeric(df["temperature"], errors="coerce")

    print("NULL до dropna:")
    print(df.isna().sum())

    df = df.dropna()

    df["date"] = df["time"].dt.date
    df["hour"] = df["time"].dt.hour
    df["latitude"] = data["latitude"]
    df["longitude"] = data["longitude"]

    print("Размер после очистки:", df.shape)
    print("Типы после преобразований:")
    print(df.dtypes)
    print("NULL после очистки:")
    print(df.isna().sum())
    print(df.head())

    NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    output_path = NORMALIZED_DIR / f"normalized_{timestamp}.csv"

    df.to_csv(output_path, index=False)
    print("Normalized сохранен:", output_path)


if __name__ == "__main__":
    main()

Текущая рабочая папка: /Users/egorermolaev/Downloads/end-to-end-project-main 8/notebooks
Корень проекта: /Users/egorermolaev/Downloads/end-to-end-project-main 8
RAW_DIR: /Users/egorermolaev/Downloads/end-to-end-project-main 8/data/raw/variant_03
NORMALIZED_DIR: /Users/egorermolaev/Downloads/end-to-end-project-main 8/data/normalized/variant_03
Использую raw file: /Users/egorermolaev/Downloads/end-to-end-project-main 8/data/raw/variant_03/2026-04-13_22-55-02.json
Исходный размер: (48, 2)
Исходные типы:
time               str
temperature    float64
dtype: object
NULL до dropna:
time           0
temperature    0
dtype: int64
Размер после очистки: (48, 6)
Типы после преобразований:
time           datetime64[us]
temperature           float64
date                   object
hour                    int32
latitude              float64
longitude             float64
dtype: object
NULL после очистки:
time           0
temperature    0
date           0
hour           0
latitude       0
longitude      